In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch import nn
import torch
from torch.utils.data import TensorDataset, DataLoader

In [17]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [18]:
train_df.shape

(42000, 785)

In [19]:
X = train_df.drop("label", axis=1).values
y = train_df["label"].values

In [20]:
X = X/255
test_df = test_df.values/255

In [21]:
X = X.reshape(-1, 1, 28, 28)
test_df = test_df.reshape(-1, 1, 28, 28)

In [22]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.from_numpy(X_train).float()
X_val = torch.from_numpy(X_val).float()
y_train = torch.from_numpy(y_train).long().squeeze()
y_val = torch.from_numpy(y_val).long().squeeze()

In [23]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.Linear(128, 10),
        )

    def forward(self, X):
        return self.model(X)




In [24]:
model = CNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

In [26]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    acc = correct / total
    print(f"Epoch {epoch} | Loss {loss:.4f} | Val Acc {acc:.4f}")

Epoch 0 | Loss 0.1181 | Val Acc 0.9780
Epoch 1 | Loss 0.0717 | Val Acc 0.9810
Epoch 2 | Loss 0.0105 | Val Acc 0.9843
Epoch 3 | Loss 0.0146 | Val Acc 0.9827
Epoch 4 | Loss 0.0218 | Val Acc 0.9857
Epoch 5 | Loss 0.0009 | Val Acc 0.9873
Epoch 6 | Loss 0.0022 | Val Acc 0.9882
Epoch 7 | Loss 0.0045 | Val Acc 0.9861
Epoch 8 | Loss 0.0116 | Val Acc 0.9880
Epoch 9 | Loss 0.0158 | Val Acc 0.9885


In [27]:
test_tensor = torch.tensor(test_df, dtype=torch.float32)
with torch.no_grad():
    logits = model(test_tensor)
    preds = torch.argmax(logits, dim=1)
submission = pd.DataFrame({
    "ImageId": range(1,len(preds)+1),
    "Label": preds
})
submission.to_csv("submission.csv", index=False)